# LIME and SHAP Tutorial on a Neural Network: Explaining Diabetes-Risk Predictions

This notebook applies the same explanation methods and analyses as `exercise_week_2_shap.ipynb`, but to the neural network from `NeuNetworkDiab.ipynb` instead of a Random Forest on the Adult dataset.

**LIME** (Local Interpretable Model-agnostic Explanations) fits a simple linear model in the neighbourhood of a single prediction.

**SHAP** (SHapley Additive exPlanations) attributes each prediction to its features using Shapley values from cooperative game theory.

**What changes compared to the Random Forest notebook:**
- The model is a PyTorch neural network (21 → 32 → 16 → 1) trained on standardized features, so we wrap it in a `predict_proba` function that takes the **original (unscaled)** features. This keeps all explanations in readable units (e.g. `BMI > 33` instead of `BMI > 0.52`).
- `shap.TreeExplainer` only works for tree models. For neural networks we use **`shap.DeepExplainer`** (DeepSHAP), which is the fast, model-specific counterpart: it propagates contributions backwards through the network using DeepLIFT rules.

**Key differences at a glance:**
- LIME is **stochastic** — it samples a neighbourhood. DeepExplainer is **deterministic for a fixed background sample**.
- SHAP values **sum exactly** to `f(x) − E[f(background)]`; LIME weights are approximations.
- Unlike TreeExplainer, DeepExplainer gives an **approximation** of Shapley values, and the baseline depends on the chosen background data.

**Dataset:** BRFSS 2015 diabetes health indicators (50/50 split) — predict `Diabetes_binary` from 21 health-indicator features.

**Outline:**
1. Load & explore the data
2. Pre-process (same split and scaling as `NeuNetworkDiab.ipynb`)
3. Train the neural network
4. Build the LIME explainer
5. Explain a single prediction with LIME
6. LIME: no-diabetes prediction for contrast
7. LIME: stability across random seeds
8. LIME: aggregate weights for a pseudo-global view
9. SHAP: single-instance and global explanations
10. LIME vs SHAP — side-by-side comparison

## 0. Install dependencies

Run the cell below once if the packages are not yet installed.

In [ ]:
%pip install lime shap torch scikit-learn pandas numpy matplotlib scipy --quiet

## 1. Load & explore the data

Place `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` in the same folder as this notebook (the same file used in `NeuNetworkDiab.ipynb`).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

df = pd.read_csv("diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

print(df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
# Class balance
print(df['Diabetes_binary'].value_counts(normalize=True).round(3))

## 2. Pre-process

We use exactly the same split and scaling as `NeuNetworkDiab.ipynb`:
- 20% held out as an untouched test set
- 20% of the remainder used as a validation set for early stopping
- `StandardScaler` fitted on the training data only

The difference is that we **keep an unscaled copy** of each split (`X_train_raw`, `X_test_raw`, ...). LIME samples its neighbourhood in this original feature space, and the plots show original values.

LIME also needs to know which columns are categorical. All features in this dataset are already numeric, but the **binary** features (e.g. `HighBP`, `Smoker`, `Sex`) are really yes/no categories, so we mark them as categorical. Ordinal features such as `Age`, `GenHlth`, `Education` and `Income` are kept numeric.

In [ ]:
TARGET = 'Diabetes_binary'
FEATURE_COLS = [c for c in df.columns if c != TARGET]
class_names = ['No diabetes', 'Diabetes']
pos_label = 1   # index of the 'Diabetes' class

X = df[FEATURE_COLS].to_numpy(dtype=np.float32)
y = df[TARGET].to_numpy(dtype=np.float32)

# Keep 20% completely separate for final testing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Make a validation set from the remaining data
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.20, random_state=SEED, stratify=y_train
)

# Standardize using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

print(f'Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# Binary (0/1) features are treated as categorical by LIME
cat_cols = [c for c in FEATURE_COLS
            if set(np.unique(X_train_raw[:, FEATURE_COLS.index(c)])) <= {0.0, 1.0}]
num_cols = [c for c in FEATURE_COLS if c not in cat_cols]
cat_indices = [FEATURE_COLS.index(c) for c in cat_cols]

print('Categorical (binary):', cat_cols)
print('Numerical / ordinal :', num_cols)

## 3. Train the neural network

Same architecture, loss, optimizer and early stopping as `NeuNetworkDiab.ipynb`.

In [ ]:
# Convert data to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)

batch_size = 256
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)

In [ ]:
class DiabetesNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)   # returns logits


model = DiabetesNN(n_features=X_train.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Train the model with simple early stopping
max_epochs = 100
patience = 5

best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_state = None

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(max_epochs):
    # ----- Training -----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
    train_loss, train_accuracy = running_loss / total, correct / total

    # ----- Validation -----
    model.eval()
    running_val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            running_val_loss += loss.item() * X_batch.size(0)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
    val_loss, val_accuracy = running_val_loss / total, correct / total

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accuracies.append(train_accuracy); val_accuracies.append(val_accuracy)

    print(f'Epoch {epoch + 1:02d} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | '
          f'train acc: {train_accuracy:.3f} | val acc: {val_accuracy:.3f}')

    # ----- Early stopping -----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping at epoch {epoch + 1}')
            break

# Use the weights from the epoch with the lowest validation loss
model.load_state_dict(best_model_state)
model.eval()

In [ ]:
# Learning curves
epochs_run = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_run, train_losses, label='Train loss')
axes[0].plot(epochs_run, val_losses, label='Validation loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].plot(epochs_run, train_accuracies, label='Train accuracy')
axes[1].plot(epochs_run, val_accuracies, label='Validation accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.show()

### 3.1 A `predict_proba` wrapper for the explainers

LIME (and model-agnostic SHAP explainers) need a function that takes a NumPy array and returns class probabilities, like scikit-learn's `predict_proba`.
The wrapper below takes **unscaled** features, applies the fitted scaler, runs the network, applies the sigmoid, and returns an `(n, 2)` array `[P(no diabetes), P(diabetes)]`.

In [ ]:
def predict_proba(X_raw_array):
    X_scaled = scaler.transform(np.asarray(X_raw_array, dtype=np.float32)).astype(np.float32)
    model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(X_scaled))).numpy().ravel()
    return np.column_stack([1 - p, p])

def predict(X_raw_array):
    return (predict_proba(X_raw_array)[:, 1] >= 0.5).astype(int)

test_proba = predict_proba(X_test_raw)[:, 1]
test_pred = predict(X_test_raw)

print(f'Test accuracy: {accuracy_score(y_test, test_pred):.3f}')
print(f'ROC-AUC:       {roc_auc_score(y_test, test_proba):.3f}\n')
print(classification_report(y_test, test_pred, target_names=class_names, digits=3))

## 4. Build the LIME Explainer

`LimeTabularExplainer` needs:
- `training_data` — to learn the feature distributions for sampling (here: the **unscaled** training data)
- `feature_names` — column names
- `class_names` — label names
- `categorical_features` — indices of the binary columns
- `categorical_names` — mapping from encoded integer → readable string (for display)

In [ ]:
import lime
import lime.lime_tabular

# Readable names for the binary features (0 = No, 1 = Yes; for Sex: 0 = female, 1 = male)
categorical_names = {}
for col in cat_cols:
    col_idx = FEATURE_COLS.index(col)
    categorical_names[col_idx] = ['Female', 'Male'] if col == 'Sex' else ['No', 'Yes']

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_raw,
    feature_names=FEATURE_COLS,
    class_names=class_names,
    categorical_features=cat_indices,
    categorical_names=categorical_names,
    mode='classification',
    random_state=42,
)
print('Explainer ready.')

## 5. Explain a single prediction

Pick one test instance and see *why* the network made its prediction.

In [ ]:
# Pick the first test instance predicted as Diabetes
diabetes_mask = test_pred == pos_label
idx = np.where(diabetes_mask)[0][0]

instance = X_test_raw[idx]
pred_proba = predict_proba(instance.reshape(1, -1))[0]
pred_class = class_names[int(pred_proba.argmax())]

print(f'Explaining instance index {idx}')
print(f'Predicted class : {pred_class}   (true class: {class_names[int(y_test[idx])]})')
print(f'Class probabilities: {dict(zip(class_names, pred_proba.round(3)))}')
pd.Series(instance, index=FEATURE_COLS).to_frame('value').T

In [ ]:
exp = explainer.explain_instance(
    data_row=instance,
    predict_fn=predict_proba,
    num_features=10,      # top-10 features in the explanation
    num_samples=5000,     # neighbourhood samples for the surrogate
)

fig = exp.as_pyplot_figure(label=pos_label)
fig.suptitle(f'LIME explanation — predicted: {pred_class} ({pred_proba.max():.1%})',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Reading the plot

- **Green bars** push the prediction *toward* `Diabetes`; **red bars** push it *away*.
- Each bar is labelled with the feature condition for *this specific instance* (e.g. `HighBP=Yes`, `BMI > 33.00`).
- The x-axis is the weight of that feature in the local linear surrogate — not a global feature importance.

In [ ]:
# Also inspect the raw weights as a DataFrame
weights = exp.as_list(label=pos_label)
pd.DataFrame(weights, columns=['feature condition', 'LIME weight']).sort_values(
    'LIME weight', key=abs, ascending=False
)

## 6. Explain a no-diabetes prediction for contrast

In [ ]:
no_diabetes_mask = test_pred == 0
idx2 = np.where(no_diabetes_mask)[0][0]

instance2 = X_test_raw[idx2]
pred_proba2 = predict_proba(instance2.reshape(1, -1))[0]
pred_class2 = class_names[int(pred_proba2.argmax())]

exp2 = explainer.explain_instance(
    data_row=instance2,
    predict_fn=predict_proba,
    num_features=10,
    num_samples=5000,
)

print(f'Explaining instance index {idx2} (true class: {class_names[int(y_test[idx2])]})')
fig2 = exp2.as_pyplot_figure(label=pos_label)
fig2.suptitle(f'LIME explanation — predicted: {pred_class2} ({pred_proba2.max():.1%})',
              fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Stability: same instance, different random seeds

LIME is stochastic — the neighbourhood is sampled randomly.
Let's check how stable the top features are across seeds.

In [ ]:
from collections import Counter

N_RUNS = 10
top_k = 5

feature_counts = Counter()
for seed in range(N_RUNS):
    local_exp = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_raw,
        feature_names=FEATURE_COLS,
        class_names=class_names,
        categorical_features=cat_indices,
        categorical_names=categorical_names,
        mode='classification',
        random_state=seed,
    ).explain_instance(
        data_row=instance,
        predict_fn=predict_proba,
        num_features=top_k,
        num_samples=5000,
    )
    for feat_name, _ in local_exp.as_list(label=pos_label):
        feature_counts[feat_name] += 1

stability_df = pd.DataFrame(
    feature_counts.most_common(), columns=['feature condition', f'appearances (/{N_RUNS})']
)
print(stability_df.to_string(index=False))

Features appearing in all 10 runs are the most reliable parts of the explanation.
If a feature only appears in 2–3 runs, treat it with caution.

## 8. Aggregate LIME weights over many test instances

LIME is designed for *local* explanations, but averaging weights over many instances gives a rough global picture of which features matter most.

Note: we map weights to features with `exp.as_map()`, which returns `(feature index, weight)` pairs. Matching on the condition string would miss conditions that start with a number, such as `24.00 < BMI <= 27.00`.

In [ ]:
N_EXPLAIN = 100  # increase for smoother averages (slower)
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(X_test_raw), size=N_EXPLAIN, replace=False)

weight_accumulator = {f: [] for f in FEATURE_COLS}

for i in sample_idx:
    e = explainer.explain_instance(
        data_row=X_test_raw[i],
        predict_fn=predict_proba,
        num_features=len(FEATURE_COLS),
        num_samples=1000,   # fewer samples → faster but noisier
    )
    for feat_idx, w in e.as_map()[pos_label]:
        weight_accumulator[FEATURE_COLS[feat_idx]].append(w)

mean_abs_weight = {
    f: np.mean(np.abs(ws)) for f, ws in weight_accumulator.items() if ws
}
global_df = pd.DataFrame(
    sorted(mean_abs_weight.items(), key=lambda x: x[1], reverse=True),
    columns=['feature', 'mean |LIME weight|']
)
print(global_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(global_df['feature'][::-1], global_df['mean |LIME weight|'][::-1], color='steelblue')
ax.set_xlabel(f'Mean |LIME weight| over {N_EXPLAIN} test instances')
ax.set_title('Approximate global feature importance via LIME')
plt.tight_layout()
plt.show()

## 9. SHAP: interpreting the same model differently

**SHAP** (SHapley Additive exPlanations) computes each feature's contribution using Shapley values from cooperative game theory.

For a Random Forest we could use `TreeExplainer`. That does not work for a neural network, so we use **`shap.DeepExplainer`** (DeepSHAP):
- It combines Shapley values with **DeepLIFT**: contributions are propagated backwards through the layers, comparing each activation with its value for a set of **background** (reference) samples.
- It needs the model itself (PyTorch) and a background sample of inputs in the **scaled** space the network was trained on.
- To make SHAP values comparable with LIME (which explains probabilities), we append a `Sigmoid` so the explained output is `P(Diabetes)` instead of the logit.

**Key properties:**
- **Efficiency** — SHAP values sum to `f(x) − E[f(background)]`: the gap between the prediction and the average prediction on the background data.
- **Deterministic** — for a fixed background sample there is no sampling noise. But a different background gives a different baseline and (slightly) different values.
- **Approximate** — DeepSHAP is a fast approximation of Shapley values, not the exact values that TreeExplainer gives for trees.

In [ ]:
import shap

# Explain the probability output: logits → sigmoid
prob_model = nn.Sequential(model, nn.Sigmoid()).eval()

# Background (reference) data: a random sample of the scaled training data
N_BACKGROUND = 200
rng_bg = np.random.default_rng(SEED)
bg_idx = rng_bg.choice(len(X_train), size=N_BACKGROUND, replace=False)
background = torch.tensor(X_train[bg_idx], dtype=torch.float32)

deep_explainer = shap.DeepExplainer(prob_model, background)

def to_2d(shap_vals):
    # DeepExplainer returns a list, a 2-D or a 3-D array depending on shap version
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[0]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 0]
    return shap_vals

# SHAP values for a 500-instance sample of the test set
N_SHAP = 500
sv_pos_all = to_2d(deep_explainer.shap_values(torch.tensor(X_test[:N_SHAP])))

expected_val = float(np.ravel(np.asarray(deep_explainer.expected_value))[0])

print(f'SHAP values shape : {sv_pos_all.shape}')
print(f'Baseline (E[f(X)]): {expected_val:.4f}')

In [ ]:
# Check the efficiency property: baseline + sum of SHAP values ≈ predicted probability
f_x = predict_proba(X_test_raw[:N_SHAP])[:, 1]
reconstructed = expected_val + sv_pos_all.sum(axis=1)
print(f'Max |f(x) − (E[f(X)] + Σ SHAP)| over {N_SHAP} instances: {np.max(np.abs(f_x - reconstructed)):.2e}')

### 9.1 Single-instance explanations with SHAP

A **waterfall plot** starts from the baseline `E[f(X)]` and stacks each feature's contribution until it reaches the model's prediction `f(x)`.

- **Red bars** push the prediction *higher* (toward `Diabetes`).
- **Blue bars** push it *lower*.
- The bars sum to `f(x) − E[f(X)]`.

The values shown next to the feature names are the original (unscaled) feature values.
Compare these with the LIME bar charts in sections 5–6 — same instance, same model, different method.

In [ ]:
sv_hi_pos = to_2d(deep_explainer.shap_values(torch.tensor(X_test[idx:idx + 1])))[0]

shap_exp_hi = shap.Explanation(
    values=sv_hi_pos,
    base_values=expected_val,
    data=instance,              # unscaled values for display
    feature_names=FEATURE_COLS,
)
print(f'Instance predicted as: {pred_class} ({pred_proba.max():.1%})')
shap.plots.waterfall(shap_exp_hi)

In [ ]:
sv_lo_pos = to_2d(deep_explainer.shap_values(torch.tensor(X_test[idx2:idx2 + 1])))[0]

shap_exp_lo = shap.Explanation(
    values=sv_lo_pos,
    base_values=expected_val,
    data=instance2,
    feature_names=FEATURE_COLS,
)
print(f'Instance predicted as: {pred_class2} ({pred_proba2.max():.1%})')
shap.plots.waterfall(shap_exp_lo)

### 9.2 Global feature importance: the SHAP summary plot

Aggregating SHAP values over many instances gives a consistent global picture.

A **beeswarm plot** shows every instance as a dot. The x-axis is the SHAP value; colour encodes the raw feature value (red = high, blue = low). This reveals not just *which* features matter, but *how* their values relate to predictions — something a simple bar chart can't show.

In [ ]:
shap.summary_plot(sv_pos_all, X_test_raw[:N_SHAP], feature_names=FEATURE_COLS)

In [ ]:
mean_abs_shap = np.mean(np.abs(sv_pos_all), axis=0)
shap_global_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean |SHAP|': mean_abs_shap,
}).sort_values('mean |SHAP|', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(shap_global_df['feature'][::-1], shap_global_df['mean |SHAP|'][::-1], color='#e06c00')
ax.set_xlabel(f'Mean |SHAP value| over {N_SHAP} test instances')
ax.set_title('Global feature importance via SHAP (DeepExplainer)')
plt.tight_layout()
plt.show()
print(shap_global_df.to_string(index=False))

## 10. LIME vs SHAP — side-by-side comparison

Both methods attribute a prediction to its features, but via different mechanisms. Here we compare them directly on **the same instance** and on the **same global ranking**.

In [ ]:
# LIME weights for the diabetes instance → map to feature names
lime_by_feature = {FEATURE_COLS[i]: w for i, w in exp.as_map()[pos_label]}

# SHAP values for the same instance
shap_by_feature = dict(zip(FEATURE_COLS, sv_hi_pos))

# Compare over the top-10 features by mean |SHAP| (global ranking)
top10 = shap_global_df['feature'].head(10).tolist()
lime_local = [lime_by_feature.get(f, 0) for f in top10]
shap_local = [shap_by_feature[f] for f in top10]

x = np.arange(len(top10))
bar_w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(x + bar_w/2, lime_local, bar_w, label='LIME weight', color='steelblue', alpha=0.85)
ax.barh(x - bar_w/2, shap_local, bar_w, label='SHAP value', color='#e06c00', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(top10)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Attribution value')
ax.set_title(f'LIME vs SHAP — same instance (predicted: {pred_class})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

# Align global rankings by feature name
lime_series = global_df.set_index('feature')['mean |LIME weight|']
shap_series = shap_global_df.set_index('feature')['mean |SHAP|']
aligned = pd.DataFrame({'LIME': lime_series, 'SHAP': shap_series}).dropna()

rho, pval = spearmanr(aligned['LIME'], aligned['SHAP'])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(aligned['LIME'], aligned['SHAP'], color='purple', alpha=0.75, zorder=3)
for feat, row in aligned.iterrows():
    ax.annotate(feat, (row['LIME'], row['SHAP']), fontsize=7.5, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Mean |LIME weight|')
ax.set_ylabel('Mean |SHAP value|')
ax.set_title(f'Global importance: LIME vs SHAP\nSpearman ρ = {rho:.2f}  (p = {pval:.3f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Spearman rank correlation: ρ = {rho:.3f},  p = {pval:.4f}')

 **Exercise:** Spend some time testing the robustness of LIME and SHAP and comparing their respective explanations:

1. Try to resample the 100 samples used for the global LIME plot. Does the explanation change?
2. Try using different surrogate models for LIME (the `model_regressor` argument of `explain_instance`). How do the explanations change? Are they consistent? Which is easier to interpret?
3. Rerun SHAP with different implementations, e.g. `shap.GradientExplainer(prob_model, background)` or the model-agnostic `shap.KernelExplainer(lambda X: predict_proba(X)[:, 1], shap.kmeans(X_train_raw, 50))` (use only a few instances — it is slow). Also try a different size or composition of the background sample for `DeepExplainer`. Do the explanations change?
4. SHAP and LIME highlight different features as being the most important. Can you do anything to assess which model is more correct?
5. Retrain the neural network with a different seed or architecture that reaches similar test accuracy. Do the explanations stay the same?